# Анализ блочной дельта-настройки коэффициентов PID

Поколения v3.x–v5.x: агент наблюдает статистику ошибки за блок и выдаёт **приращения** коэффициентов $(\Delta K_p, \Delta K_i, \Delta K_d)$.

**Группа A** (`td3_train_async`/`td3_inference_async`, v3.x): конфиг в `.hydra/`, env.log — блочные сводки с колонкой `phase`, switch-to-NN из `phase`.

**Группа B** (`pid_delta_tuning*`, v4.x–v5.x): конфиг в `config.yaml`, env.log — строки `step`/`reset` (`should_reset`), всё нормированное, switch-to-NN из конфига.

Эксперименты:
* `2025-10-28_19-57-02_td3_train_async` — Kp (v3.1.0)
* `2025-10-30_17-30-32_td3_train_async` — Kp, Ki (v3.1.2)
* `2025-11-05_15-54-48_td3_train_async` — Kp, Ki, Kd (v3.1.3)
* `2025-11-06_14-26-53_td3_train_async` — Kp, Ki, сброс напряжения на середину (v3.1.4)
* `2025-11-07_18-53-51_td3_train_async` — Kp, Ki, нижняя граница 200
* `2025-11-11_15-03-10_td3_inference_async` — Kp, Ki, **инференс** (v3.1.5)
* `2025-11-25_15-51-58_pid_delta_tuning` — Kp, Ki (v4.0.0)
* `2025-11-26_15-53-18_pid_delta_tuning` — Kp, Ki (v4.0.0, повтор)
* `2025-11-28_16-15-57_pid_delta_tuning-inference` — Kp, Ki, **инференс** (v4.0.1)
* `2025-12-02_12-50-33_pid_delta_tuning` — Kp, Ki, Kd (v4.1.1)
* `2025-12-04_16-12-21_pid_delta_tuning`, `2025-12-04_17-21-28_pid_delta_tuning` — Kp, Ki, Kd (v4.1.2)
* `2025-12-09_16-31-56_pid_delta_tuning-recurrent` — Kp, Ki, **LSTM** (v4.1.4)
* `2026-01-12_19-41-32_pid_delta_tuning` — Kp, Ki, Kd, термостабилизация (v5.0.2)
* `2026-01-17_15-18-43_pid_delta_tuning` и серия `2026-01-19_*_pid_delta_tuning` — Kp, Ki, Kd, автоопределение setpoint, развёртка factor (v5.0.4)

**Презентационные графики** (последняя секция) в оригинальных блокнотах делались на прогонах группы B:
* combo «коэффициенты + СКО ошибки» и PV+CO — `2025-11-25_15-51-58_pid_delta_tuning` (Kp, Ki) и `2025-11-28_16-15-57pid_delta_tuning-inference` (инференс);
* те же для 3 коэффициентов — `2025-12-02_12-50-33_pid_delta_tuning` / `2025-12-04_16-12-21_pid_delta_tuning` (предзащита);
* 5-панельная фигура для ICLO — `2026-01-17_15-18-43_pid_delta_tuning`.

Перед построением презентационных графиков загрузите соответствующий эксперимент.

In [ ]:
%load_ext autoreload
%autoreload 2

## Описание эксперимента

### Алгоритм
**TD3** (MLP; рекуррентный LSTM — для `*-recurrent`). Сбор данных и обучение разделены. Шум исследования к действиям не подмешивается — разнообразие данных обеспечивает предобучение (случайные коэффициенты в фазе `pretrain` у группы A; начальный сбор у группы B).

### Окружение
Один шаг агента = блок сырых шагов (первые `burn_in` отбрасываются, по остальным считается статистика).

- **Состояние:** $[\text{error\_mean\_norm},\ \text{error\_std\_norm}]$ — статистика ошибки $\text{PV}-\text{setpoint}$ за блок.
- **Действие:** приращения $(\Delta K_p, \Delta K_i, \Delta K_d)$; применяются к текущим коэффициентам (а не задаются абсолютно, как в `pid_block_coefficient_tuning`).
- **Награда** логируется (реконструировать не нужно). У группы B — взвешенная (`precision` + `stability` + `action`).

### Конечность диапазона напряжения
- **Группа A** (`ControlLimitManager`): при выходе control output за порог пределы ЦАП на `enforcement_steps` зажимаются в окно у середины (`force_min..force_max`), затем отпускаются. Эпизод не завершается.
- **Группа B** (`should_reset`): при выходе среднего за блок control output за `[min_threshold, max_threshold]` эпизод **терминируется**, напряжение принудительно возвращается в середину `[2000, 2500]`. В env-логе — строки `reset` и `should_reset=True`.

### Длительность: эксперимента vs обучения
Различаем **длительность эксперимента** (весь прогон от старта сбора данных) и **длительность обучения** (с момента `Training started` — когда накоплено достаточно данных, `min_data_for_training` / `initial_collect`). Сбор данных здесь длинный (предобучение/начальный сбор), поэтому эти величины заметно отличаются.

## Загрузка данных

In [ ]:
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Optional
import re

import numpy as np
import pandas as pd
from omegaconf import DictConfig, OmegaConf

from nn_laser_stabilizer.utils.paths import get_experiment_dir
from nn_laser_stabilizer.analysis.sources import (
    parse_keyval_log, parse_train_log, parse_connection_log,
)

# Группа A: структурные .log, конфиг в .hydra/. Остальные имена — группа B.
GROUP_A_NAMES = {"td3_train_async", "td3_inference_async"}


@dataclass
class ExperimentData:
    env_df: pd.DataFrame
    connection_df: pd.DataFrame
    train_df: Optional[pd.DataFrame]    # None для инференс-прогонов
    experiment_duration_sec: float      # весь прогон (включая сбор данных)
    training_duration_sec: Optional[float]  # с 'Training started'; None у инференса
    neural_network_step: int            # шаг, на котором агент берёт управление
    reset_steps: list                   # шаги reset (группа B); [] у группы A
    config: DictConfig
    fmt: str                            # "A" | "B"


def _load_env_a(exp_dir: Path) -> pd.DataFrame:
    # env.log: блочные сводки со столбцом phase; имена уже snake_case.
    raw = parse_keyval_log(exp_dir / "env_logs" / "env.log")
    cols = [c for c in ["step", "phase", "kp", "ki", "kd", "delta_kp",
                        "error_mean", "error_std", "error_mean_norm",
                        "error_std_norm", "reward"] if c in raw.columns]
    return raw[cols].reset_index(drop=True)


def _load_env_b(exp_dir: Path) -> pd.DataFrame:
    # env.log: строки step (есть should_reset) и reset (нет). reset наследует
    # номер последнего шага через ffill.
    raw = parse_keyval_log(exp_dir / "env_logs" / "env.log")
    is_step = raw["should_reset"].notna() if "should_reset" in raw.columns else raw["step"].notna()
    raw = raw.copy()
    raw["type"] = np.where(is_step, "step", "reset")
    raw["step"] = raw["step"].ffill().fillna(0).astype(int)
    cols = [c for c in ["step", "time", "type", "kp", "ki", "kd",
                        "delta_kp_norm", "delta_ki_norm", "delta_kd_norm", "delta_kd",
                        "error_mean_norm", "error_std_norm", "reward", "should_reset"]
            if c in raw.columns]
    return raw[cols].reset_index(drop=True)


def _load_connection(exp_dir: Path) -> pd.DataFrame:
    # i-я посылка (SEND) спаривается с i-м чтением (READ).
    raw = parse_connection_log(exp_dir / "connection_logs" / "connection.log")
    event = raw["event"].astype(str).str.upper()
    send = raw[event == "SEND"][
        [c for c in ["kp", "ki", "kd", "control_min", "control_max"] if c in raw.columns]
    ].reset_index(drop=True)
    send["step"] = range(len(send))
    read = raw[event == "READ"][
        [c for c in ["process_variable", "control_output"] if c in raw.columns]
    ].reset_index(drop=True)
    read["step"] = range(len(read))
    return send.merge(read, on="step", how="left")


_LOG_TS_RE = re.compile(r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})(?:,(\d{3}))?\]")


def _parse_log_time(line: str):
    m = _LOG_TS_RE.match(line)
    if not m:
        return None
    dt = datetime.strptime(m.group(1), "%Y-%m-%d %H:%M:%S")
    if m.group(2):
        dt = dt.replace(microsecond=int(m.group(2)) * 1000)
    return dt


def _durations_from_main_log(exp_dir: Path) -> tuple[float, Optional[float]]:
    """Возвращает (длительность эксперимента, длительность обучения).

    Эксперимент: первая -> последняя метка времени (весь прогон, включая сбор
    данных). Обучение: 'Training started' -> 'Training finished' (обучение
    начинается, когда накоплено достаточно данных). У инференс-прогонов
    обучения нет -> None.
    """
    logs = [p for p in exp_dir.glob("*.log")]
    if not logs:
        raise ValueError(f"В {exp_dir} нет main-лога (*.log)")
    first = last = train_start = finished = None
    with open(logs[0], encoding="utf-8", errors="replace") as f:
        for line in f:
            t = _parse_log_time(line)
            if t is None:
                continue
            if first is None:
                first = t
            last = t
            low = line.lower()
            if "training started" in low and train_start is None:
                train_start = t
            if "finished" in low:  # 'Training finished' (A) или 'Experiment finished' (B)
                finished = t
    if first is None or last is None:
        raise ValueError(f"В логе {logs[0]} нет меток времени")
    experiment_sec = (last - first).total_seconds()
    train_end = finished if finished is not None else last
    training_sec = (
        (train_end - train_start).total_seconds()
        if train_start is not None
        else None
    )
    return experiment_sec, training_sec


def load_experiment(
    experiment_name: str, experiment_date: str, experiment_time: str
) -> ExperimentData:
    exp_dir = get_experiment_dir(
        experiment_name=experiment_name,
        experiment_date=experiment_date,
        experiment_time=experiment_time,
    )

    fmt = "A" if experiment_name in GROUP_A_NAMES else "B"

    if fmt == "A":
        config = OmegaConf.load(exp_dir / ".hydra" / "config.yaml")
        env_df = _load_env_a(exp_dir)
        normal = env_df[env_df["phase"] == "normal"] if "phase" in env_df.columns else env_df.iloc[0:0]
        neural_network_step = int(normal["step"].iloc[0]) if len(normal) else int(env_df["step"].min())
        reset_steps = []
        train_path = exp_dir / "train_logs" / "train.log"
    else:
        config = OmegaConf.load(exp_dir / "config.yaml")
        env_df = _load_env_b(exp_dir)
        init = OmegaConf.select(config, "training.initial_collect_steps", default=0)
        expl = OmegaConf.select(config, "training.exploration_steps", default=0)
        neural_network_step = int(max(init, expl))
        reset_steps = env_df.loc[env_df.get("type") == "reset", "step"].tolist() if "type" in env_df.columns else []
        train_path = exp_dir / "logs" / "train.log"

    train_df = None
    if train_path.exists():
        train_df = parse_train_log(train_path).rename(
            columns={"Loss/Critic": "critic_loss", "Loss/Actor": "actor_loss"}
        )

    connection_df = _load_connection(exp_dir)
    experiment_duration_sec, training_duration_sec = _durations_from_main_log(exp_dir)

    return ExperimentData(
        env_df=env_df,
        connection_df=connection_df,
        train_df=train_df,
        experiment_duration_sec=experiment_duration_sec,
        training_duration_sec=training_duration_sec,
        neural_network_step=neural_network_step,
        reset_steps=reset_steps,
        config=config,
        fmt=fmt,
    )

In [ ]:
data = load_experiment(
    experiment_name="pid_delta_tuning",
    experiment_date="2025-12-02",
    experiment_time="12-50-33",
)

print(f"Формат: {data.fmt}")
print(f"Длительность эксперимента: {data.experiment_duration_sec / 60:.1f} мин")
if data.training_duration_sec is not None:
    print(f"Длительность обучения: {data.training_duration_sec / 60:.1f} мин")
else:
    print("Обучение: нет (инференс)")
print(f"Switch to NN (шаг): {data.neural_network_step}")
print(f"Reset-событий: {len(data.reset_steps)}")
print(f"env_df: {len(data.env_df)} строк; колонки: {list(data.env_df.columns)}")
print(f"connection_df: {len(data.connection_df)} строк")
print(f"train_df: {None if data.train_df is None else len(data.train_df)}")

In [ ]:
print(OmegaConf.to_yaml(data.config))

In [ ]:
import matplotlib.pyplot as plt
from functools import wraps

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

if not getattr(plt.savefig, "_auto_wrapped", False):
    _original_savefig = plt.savefig

    @wraps(_original_savefig)
    def _auto_savefig(filename, *args, **kwargs):
        filename = OUTPUT_DIR / Path(filename).name
        return _original_savefig(filename, *args, **kwargs)

    setattr(_auto_savefig, "_auto_wrapped", True)
    plt.savefig = _auto_savefig

## Анализ окружения

In [ ]:
env_df = data.env_df

# Только строки-шаги (у группы B reset-строки содержат NaN-метрики).
if "type" in env_df.columns:
    env_steps = env_df[env_df["type"] == "step"].copy()
else:
    env_steps = env_df.copy()

delta_cols = [c for c in env_steps.columns if c.startswith("delta_")]
std_col = "error_std" if "error_std" in env_steps.columns else "error_std_norm"
mean_col = "error_mean" if "error_mean" in env_steps.columns else "error_mean_norm"

print(f"Шагов: {len(env_steps)}; дельты: {delta_cols}; ошибки: {mean_col}, {std_col}")

In [ ]:
tag = "kp"
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(env_steps["step"], env_steps[tag], color="blue", linewidth=0.8, label=tag)
ax.axvline(data.neural_network_step, color="red", linestyle="--", linewidth=2, label="switch to NN")
for rs in data.reset_steps:
    ax.axvline(rs, color="orange", linestyle=":", linewidth=0.6, alpha=0.25)
ax.set_title(f"{tag} over steps", fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel(tag)
ax.legend()
plt.tight_layout()
plt.savefig(f"kp.pdf")
plt.show()

In [ ]:
tag = "ki"
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(env_steps["step"], env_steps[tag], color="orange", linewidth=0.8, label=tag)
ax.axvline(data.neural_network_step, color="red", linestyle="--", linewidth=2, label="switch to NN")
for rs in data.reset_steps:
    ax.axvline(rs, color="orange", linestyle=":", linewidth=0.6, alpha=0.25)
ax.set_title(f"{tag} over steps", fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel(tag)
ax.legend()
plt.tight_layout()
plt.savefig(f"ki.pdf")
plt.show()

In [ ]:
tag = "kd"
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(env_steps["step"], env_steps[tag], color="green", linewidth=0.8, label=tag)
ax.axvline(data.neural_network_step, color="red", linestyle="--", linewidth=2, label="switch to NN")
for rs in data.reset_steps:
    ax.axvline(rs, color="orange", linestyle=":", linewidth=0.6, alpha=0.25)
ax.set_title(f"{tag} over steps", fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel(tag)
ax.legend()
plt.tight_layout()
plt.savefig(f"kd.pdf")
plt.show()

In [ ]:
# Дельты коэффициентов (рисуем те, что есть: raw для A, _norm для B).
fig, ax = plt.subplots(figsize=(12, 5))
for c in delta_cols:
    ax.plot(env_steps["step"], env_steps[c], linewidth=0.8, label=c)
ax.axvline(data.neural_network_step, color="red", linestyle="--", linewidth=2, label="switch to NN")
ax.set_title("Deltas over steps", fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel("delta")
ax.legend()
plt.tight_layout()
plt.savefig("deltas.pdf")
plt.show()

In [ ]:
# Error mean (raw и/или norm — что присутствует).
fig, ax = plt.subplots(figsize=(12, 5))
for c in ["error_mean", "error_mean_norm"]:
    if c in env_steps.columns:
        ax.plot(env_steps["step"], env_steps[c], linewidth=0.8, label=c)
ax.axvline(data.neural_network_step, color="red", linestyle="--", linewidth=2, label="switch to NN")
for rs in data.reset_steps:
    ax.axvline(rs, color="orange", linestyle=":", linewidth=0.6, alpha=0.25)
ax.set_title("Error mean", fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel("error_mean")
ax.legend()
plt.tight_layout()
plt.savefig("error_mean.pdf")
plt.show()

In [ ]:
# Error std (raw и/или norm).
fig, ax = plt.subplots(figsize=(12, 5))
for c in ["error_std", "error_std_norm"]:
    if c in env_steps.columns:
        ax.plot(env_steps["step"], env_steps[c], linewidth=0.8, label=c)
ax.axvline(data.neural_network_step, color="red", linestyle="--", linewidth=2, label="switch to NN")
for rs in data.reset_steps:
    ax.axvline(rs, color="orange", linestyle=":", linewidth=0.6, alpha=0.25)
ax.set_title("Error std", fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel("error_std")
ax.legend()
plt.tight_layout()
plt.savefig("error_std.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(env_steps["step"], env_steps["reward"], color="green", linewidth=0.8, label="reward")
ax.axvline(data.neural_network_step, color="red", linestyle="--", linewidth=2, label="switch to NN")
for rs in data.reset_steps:
    ax.axvline(rs, color="orange", linestyle=":", linewidth=0.6, alpha=0.25)
ax.set_title("Reward", fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel("reward")
ax.legend()
plt.tight_layout()
plt.savefig("reward.pdf")
plt.show()

## Связь с железом (connection)

In [ ]:
conn = data.connection_df

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(conn["step"], conn["process_variable"], color="blue", linewidth=0.8)
ax.axhline(data.config.env.setpoint if "setpoint" in data.config.env else 1200,
           color="black", linestyle="--", label="setpoint")
ax.set_title("Process Variable", fontsize=14)
ax.set_xlabel("Exchange step")
ax.set_ylabel("process_variable")
ax.legend()
plt.tight_layout()
plt.savefig("process_variable.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(conn["step"], conn["control_output"], color="red", linewidth=0.8)
ax.set_title("Control Output", fontsize=14)
ax.set_xlabel("Exchange step")
ax.set_ylabel("control_output")
plt.tight_layout()
plt.savefig("control_output.pdf")
plt.show()

## Анализ обучения

In [ ]:
if data.train_df is None:
    print("Инференс-прогон: train-лога нет.")
else:
    fig, ax = plt.subplots(figsize=(12, 6))
    for tag in ["critic_loss", "actor_loss"]:
        if tag in data.train_df.columns:
            ax.plot(data.train_df["step"], data.train_df[tag], label=tag)
    ax.set_title("Training loss", fontsize=14)
    ax.set_xlabel("Step")
    ax.set_ylabel("Value")
    ax.legend(title="Metric")
    plt.tight_layout()
    plt.savefig("train_loss.pdf")
    plt.show()

## Диагностика

In [ ]:
# Зависимость std ошибки от Kp.
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(env_steps["kp"], env_steps[std_col], alpha=0.4, s=10)
ax.set_title(f"{std_col} vs Kp", fontsize=14)
ax.set_xlabel("kp")
ax.set_ylabel(std_col)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("kp_vs_error_std.pdf")
plt.show()

In [ ]:
# Матрица корреляций (matplotlib, без seaborn).
corr_cols = [c for c in ["kp", "ki", "kd", mean_col, std_col, "reward"] if c in env_steps.columns]
corr = env_steps[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticks(range(len(corr_cols)))
ax.set_yticklabels(corr_cols)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig("corr_heatmap.pdf")
plt.show()

## Графики для докладов

Презентационные графики перенесены **в первозданном виде** из блокнотов 16/18/19/23 (все — группа B), сгруппированы по исходному эксперименту. Перед запуском загрузите соответствующий прогон. Старые имена переменных восстанавливает ячейка-шим ниже. Некоторые ячейки переопределяют `neural_network_step` (как в оригинале) — запускайте секцию по порядку.

In [ ]:
import numpy as np

config = data.config
neural_network_step = data.neural_network_step
reset_steps = data.reset_steps
loss_df = data.train_df

initial_collect_steps = OmegaConf.select(config, "training.initial_collect_steps", default=0)
exploration_steps = OmegaConf.select(config, "training.exploration_steps", default=0)
block_size = int(OmegaConf.select(config, "env.args.block_size", default=200))

setpoint = OmegaConf.select(config, "env.setpoint", default=None)
if setpoint is None:
    setpoint = OmegaConf.select(config, "env.args.setpoint", default=1200)

# step_df с историческими ("человеческими") именами колонок.
step_df = env_steps.rename(columns={
    "kp": "Kp", "ki": "Ki", "kd": "Kd",
    "error_mean_norm": "Error mean norm", "error_std_norm": "Error std norm",
}).copy()
step_df["Step"] = step_df["step"]

# connection_df с обоими вариантами имён (snake — 16/18/19; Capitalized — 23).
connection_df = data.connection_df.copy()
connection_df["Connection step"] = connection_df["step"] * block_size
connection_df["Control output"] = connection_df["control_output"]

### Эксперимент 2025-11-25_15-51-58 (Kp, Ki)

In [ ]:
from matplotlib.gridspec import GridSpec

# Настройка
plt.rcParams.update({'font.size': 14})  # крупный шрифт для слайда

neural_network_step = max(initial_collect_steps, exploration_steps) - 100 # этап warmup

# Параметры
columns_left = ['Kp', 'Ki']
colors_left = ['green', 'blue']  # разные цвета для Kp и Ki
column_right = 'Error std norm'

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(2, 2, width_ratios=[1, 1.2], height_ratios=[1,1], wspace=0.3, hspace=0.4)

# --- Левая часть: Kp и Ki ---
for i, col in enumerate(columns_left):
    ax = fig.add_subplot(gs[i,0])
    ax.plot(step_df['Step'], step_df[col], alpha=0.8, linewidth=1.2, color=colors_left[i])
    
    if neural_network_step <= step_df['Step'].max():
        ax.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2)
    
    ax.set_ylabel(col)
    ax.set_xlabel('Block')
    ax.grid(True, alpha=0.3)

# --- Правая часть: Error std norm ---
ax_right = fig.add_subplot(gs[:,1])  # занимает обе строки
ax_right.plot(step_df['Step'], step_df[column_right], alpha=0.8, linewidth=1.5, color='purple', label=column_right)

if neural_network_step <= step_df['Step'].max():
    ax_right.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, label='Switch to NN')

ax_right.set_xlabel('Block')
ax_right.set_ylabel(column_right)
ax_right.grid(True, alpha=0.3)
ax_right.legend()

fig.subplots_adjust(left=0.08, right=0.95, top=0.93, bottom=0.08, wspace=0.3, hspace=0.4)

In [ ]:
plt.rcParams.update({
    "font.size": 24,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 24,
})

columns_left = ['Kp', 'Ki']
colors_left = ['green', 'blue']
column_right = 'Error std norm'
column_right_ru = 'СКВО напряжения\nна фотодетекторе'

time_min = (step_df['time'] - step_df['time'].min()) / 60.0

switch_time_min = None
if neural_network_step <= step_df['Step'].max():
    mask_exact = step_df['Step'] == neural_network_step
    if mask_exact.any():
        switch_time_min = float(time_min[mask_exact].iloc[0])
    else:
        switch_time_min = float(
            np.interp(neural_network_step, step_df['Step'].to_numpy(), time_min.to_numpy())
        )

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(
    2, 2,
    width_ratios=[1, 1.2],
    height_ratios=[1, 1],
    wspace=0.3, hspace=0.4
)

axes_left = []
for i, col in enumerate(columns_left):
    if i == 0:
        ax = fig.add_subplot(gs[i, 0])
    else:
        ax = fig.add_subplot(gs[i, 0], sharex=axes_left[0])
    axes_left.append(ax)

    ax.plot(time_min, step_df[col], alpha=0.8, linewidth=1.2, color=colors_left[i])

    if switch_time_min is not None:
        ax.axvline(x=switch_time_min, color='red', linestyle='--', linewidth=2)

    ax.set_ylabel(col)
    ax.grid(False)

# Подпись X только на нижнем графике
axes_left[-1].set_xlabel('Время, мин')
for ax in axes_left[:-1]:
    ax.tick_params(labelbottom=False)

# Правый большой график
ax_right = fig.add_subplot(gs[:, 1])
ax_right.plot(
    time_min,
    step_df[column_right],
    alpha=0.8,
    linewidth=1.5,
    color='purple',
)

if switch_time_min is not None:
    ax_right.axvline(
        x=switch_time_min,
        color='red',
        linestyle='--',
        linewidth=2,
        label='Переключение на НС'
    )

ax_right.set_xlabel('Время, мин')
ax_right.set_ylabel(column_right_ru)
ax_right.grid(False)

fig.align_ylabels(axes_left)
fig.subplots_adjust(left=0.08, right=0.95, top=0.93, bottom=0.10, wspace=0.3, hspace=0.4)
plt.savefig("pid_tune_2coef.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def hsl_to_rgb_normalized(h, s, l):
    # colorsys использует HLS (не HSL)
    # поэтому порядок: (h, l, s)
    from colorsys import hls_to_rgb
    return hls_to_rgb(h / 360, l / 100, s / 100)

BLUE_RGB = hsl_to_rgb_normalized(206, 73, 48)
GREEN_RGB = hsl_to_rgb_normalized(120, 72, 44)
RED_RGB = hsl_to_rgb_normalized(9, 98, 63)
PURPLE_RGB = hsl_to_rgb_normalized(279, 98, 76)
LIGHT_PURPLE_RGB = hsl_to_rgb_normalized(278, 100, 94)

GRAY_RGB = (123 / 255, 123 / 255, 123 / 255)

plt.rcParams.update({
    "font.size": 24,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 24,
})

error_mean_normalization_factor = config.env.args.error_mean_normalization_factor
error_std_normalization_factor = config.env.args.error_std_normalization_factor

error_mean = step_df['Error mean norm'] * error_mean_normalization_factor
error_std = step_df['Error std norm'] * error_std_normalization_factor
rmse = np.sqrt(error_mean ** 2 + error_std ** 2) / 10 # старый масштаб

columns_left = ['Kp', 'Ki']
colors_left = [GREEN_RGB, BLUE_RGB]
column_right_ru = 'Среднеквадратическая\nошибка'

time_min = (step_df['time'] - step_df['time'].min()) / 60.0

neural_network_step = max(initial_collect_steps, exploration_steps) - 100 # этап warmup

switch_time_min = None
if neural_network_step <= step_df['Step'].max():
    mask_exact = step_df['Step'] == neural_network_step
    if mask_exact.any():
        switch_time_min = float(time_min[mask_exact].iloc[0])
    else:
        switch_time_min = float(
            np.interp(neural_network_step, step_df['Step'].to_numpy(), time_min.to_numpy())
        )

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(
    2, 2,
    width_ratios=[1, 1.2],
    height_ratios=[1, 1],
    wspace=0.3, hspace=0.4
)

# Левый столбец: 2 графика с общей осью X
axes_left = []
for i, col in enumerate(columns_left):
    if i == 0:
        ax = fig.add_subplot(gs[i, 0])
    else:
        ax = fig.add_subplot(gs[i, 0], sharex=axes_left[0])
    axes_left.append(ax)

    ax.plot(time_min, step_df[col], alpha=0.8, linewidth=1.2, color=colors_left[i])

    if switch_time_min is not None:
        ax.axvline(x=switch_time_min, color='orange', linestyle='--', linewidth=5)

    ax.set_ylabel(col)
    ax.grid(False)

# Подпись X только на нижнем графике
axes_left[-1].set_xlabel('Время, мин')
for ax in axes_left[:-1]:
    ax.tick_params(labelbottom=False)

# Правый большой график
ax_right = fig.add_subplot(gs[:, 1])
ax_right.plot(
    time_min,
    rmse,
    alpha=0.8,
    linewidth=1.5,
    color=PURPLE_RGB,
)

if switch_time_min is not None:
    ax_right.axvline(
        x=switch_time_min,
        color='orange',
        linestyle='--',
        linewidth=5,
        label='Включение НС-контроллера'
    )

ax_right.set_xlabel('Время, мин')
ax_right.set_ylabel(column_right_ru)
ax_right.grid(False)

handles, labels = ax_right.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.95),
    ncol=2,
    frameon=False,
)

fig.align_ylabels(axes_left)
fig.subplots_adjust(left=0.08, right=0.95, top=0.93, bottom=0.10, wspace=0.3, hspace=0.4)

plt.savefig("pid_tune_2coef.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Настройка шрифта
plt.rcParams.update({'font.size': 14})

# Определяем данные
columns_left = ['process_variable']
columns_right = ['control_output']
fig = plt.figure(figsize=(18, 6))
gs = GridSpec(1, 2, width_ratios=[1, 1], wspace=0.3)

ax_left = fig.add_subplot(gs[0,0])
ax_left.plot(connection_df['step'], connection_df['process_variable'], 'b-', alpha=0.7, linewidth=1.2, label='Process Variable')
ax_left.axhline(y=setpoint, color='r', linestyle='--', label=f'Setpoint')
ax_left.set_xlim(left=100)

if neural_network_step <= connection_df['step'].max():
    ax_left.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, label='Switch to NN')

ax_left.set_xlabel('Step')
ax_left.set_ylabel('Process Variable')
ax_left.set_title('Process Variable')
ax_left.grid(True, alpha=0.3)
ax_left.legend()

ax_right = fig.add_subplot(gs[0,1])
ax_right.plot(connection_df['step'], connection_df['control_output'], 'g-', alpha=0.7, linewidth=1.2, label='Control Output')
ax_right.set_xlim(left=100)

if neural_network_step <= connection_df['step'].max():
    ax_right.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2)

ax_right.set_xlabel('Step')
ax_right.set_ylabel('Control Output')
ax_right.set_title('Control Output')
ax_right.grid(True, alpha=0.3)
ax_right.legend()

plt.tight_layout()
plt.show()


### Эксперимент 2025-11-28_16-15-57 (инференс, Kp, Ki)

In [ ]:
plt.rcParams.update({'font.size': 14})  

columns_left = ['Kp', 'Ki']
colors_left = ['green', 'blue']  
column_right = 'Error std norm'

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(2, 2, width_ratios=[1, 1.2], height_ratios=[1,1], wspace=0.3, hspace=0.4)

for i, col in enumerate(columns_left):
    ax = fig.add_subplot(gs[i,0])
    ax.plot(step_df['Step'], step_df[col], alpha=0.8, linewidth=1.2, color=colors_left[i])
    
    ax.set_ylabel(col)
    ax.set_xlabel('Block')
    ax.grid(True, alpha=0.3)

ax_right = fig.add_subplot(gs[:,1])  
ax_right.plot(step_df['Step'], step_df[column_right], alpha=0.8, linewidth=1.5, color='purple', label=column_right)


ax_right.set_xlabel('Block')
ax_right.set_ylabel(column_right)
ax_right.grid(True, alpha=0.3)
ax_right.legend()

fig.subplots_adjust(left=0.08, right=0.95, top=0.93, bottom=0.08, wspace=0.3, hspace=0.4)
plt.show()

In [ ]:
plt.rcParams.update({'font.size': 14})

setpoint = 1200

columns_left = ['process_variable']
columns_right = ['control_output']
fig = plt.figure(figsize=(18, 6))
gs = GridSpec(1, 2, width_ratios=[1, 1], wspace=0.3)

ax_left = fig.add_subplot(gs[0,0])
ax_left.plot(connection_df['step'], connection_df['process_variable'], 'b-', alpha=0.7, linewidth=1.2, label='Process Variable')
ax_left.axhline(y=setpoint, color='r', linestyle='--', label=f'Setpoint')
ax_left.set_xlim(left=100)
ax_left.set_ylim(0, 1800)

ax_left.set_xlabel('Step')
ax_left.set_ylabel('Process Variable')
ax_left.set_title('Process Variable')
ax_left.grid(True, alpha=0.3)
ax_left.legend()

ax_right = fig.add_subplot(gs[0,1])
ax_right.plot(connection_df['step'], connection_df['control_output'], 'g-', alpha=0.7, linewidth=1.2, label='Control Output')
ax_right.set_xlim(left=100)

ax_right.set_xlabel('Step')
ax_right.set_ylabel('Control Output')
ax_right.set_title('Control Output')
ax_right.grid(True, alpha=0.3)
ax_right.legend()

plt.tight_layout()
plt.show()


### Эксперименты 2025-12-02 / 2025-12-04 (Kp, Ki, Kd)

In [ ]:
cols = ['Kp', 'Ki', 'Kd']
fig, axes = plt.subplots(len(cols), 1, figsize=(14, 10), sharex=True)

for ax, col in zip(axes, cols):
    ax.plot(step_df['Step'], step_df[col], alpha=0.8, linewidth=0.8, label=col)

    for reset_step in reset_steps:
        if reset_step <= step_df['Step'].max():
            ax.axvline(x=reset_step, color='orange', linestyle=':', linewidth=2, alpha=0.6)

    if neural_network_step <= step_df['Step'].max():
        ax.axvline(
            x=neural_network_step,
            color='red',
            linestyle='--',
            linewidth=2,
            label=f'Switch NN ({neural_network_step})'
        )

    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[-1].set_xlabel("Step")
plt.suptitle("Kp / Ki / Kd over Steps", fontsize=14)
plt.tight_layout()
plt.savefig('pid_coeffs_vs_block_step.png')
plt.show()

In [ ]:
neural_network_step = neural_network_step * config.env.args.block_size + len(reset_steps) * 1000
if len(connection_df) > 0:
    setpoint = config.env.args.setpoint
    
    plt.figure(figsize=(12, 5))
    plt.plot(connection_df['step'], connection_df['process_variable'], 'b-', alpha=0.7, linewidth=0.8, label='Process Variable')
    plt.axhline(y=setpoint, color='r', linestyle='--', label=f'Setpoint ({setpoint})')
    
    for reset_step in reset_steps:
        if reset_step <= connection_df['step'].max():
            plt.axvline(x=reset_step, color='orange', linestyle=':', linewidth=2, alpha=0.6)
    
    if neural_network_step <= connection_df['step'].max():
        plt.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, 
                    label=f'Switch to NN (step {neural_network_step})')
    
    plt.title('Process Variable')
    plt.xlabel('Step')
    plt.ylim(500, 1700)
    plt.ylabel('Process Variable')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('process_variable_over_connection_steps.png')
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(connection_df['step'], connection_df['control_output'], 'g-', alpha=0.7, linewidth=0.8, label='Control Output')
    
    for reset_step in reset_steps:
        if reset_step <= connection_df['step'].max():
            plt.axvline(x=reset_step, color='orange', linestyle=':', linewidth=2, alpha=0.6)
    
    if neural_network_step <= connection_df['step'].max():
        plt.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, 
                    label=f'Switch to NN (step {neural_network_step})')
    
    plt.title('Control Output')
    plt.xlabel('Step')
    plt.ylabel('Control Output')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('control_output_over_connection_steps.png')
    plt.show()

In [ ]:
neural_network_step = neural_network_step * config.env.args.block_size + len(reset_steps) * 1000

if len(connection_df) > 0:
    setpoint = config.env.args.setpoint

    fig, ax1 = plt.subplots(figsize=(12, 5))

    # Первая ось (Process Variable)
    ax1.plot(connection_df['step'], connection_df['process_variable'],
             'g-', alpha=0.7, linewidth=0.8, label='Process Variable')
    ax1.axhline(y=setpoint, color='black', linestyle='--', alpha=0.6, linewidth=3,
                label=f'Setpoint ({setpoint})')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Process Variable', color='g')
    ax1.set_ylim(500, 1700)

    # Вторая ось (Control Output)
    ax2 = ax1.twinx()
    ax2.plot(connection_df['step'], connection_df['control_output'],
             'r-', alpha=0.7, linewidth=0.8, label='Control Output')
    ax2.set_ylabel('Control Output', color='r')

    plt.xlim(left=10)
    plt.grid(False)
    plt.tight_layout()
    plt.savefig('combined_plot.png')
    plt.show()

In [ ]:
# plt.rcParams.update({
#     "font.size": 24,
#     "axes.titlesize": 24,
#     "axes.labelsize": 24,
#     "xtick.labelsize": 24,
#     "ytick.labelsize": 24,
#     "legend.fontsize": 24,
# })

EXPERIMENT_TIME_MIN = 121

neural_network_step = neural_network_step * config.env.args.block_size + len(reset_steps) * 1000

setpoint = config.env.args.setpoint / 10

time_min = connection_df['step'] / connection_df['step'].max() * EXPERIMENT_TIME_MIN

fig, ax1 = plt.subplots(figsize=(14, 7))

# Первая ось (напряжение фотодетектора)
ax1.plot(time_min, connection_df['process_variable'] / 10,
         color='blue', alpha=0.7, linewidth=0.8)
ax1.axhline(y=setpoint, color='black', linestyle='--', linewidth=5)

ax1.set_xlabel('Время, мин')
ax1.set_ylabel('Напряжение на фотодетекторе', color='blue')
ax1.set_ylim(60, 180)

# Вторая ось (напряжение на фазовращателе)
ax2 = ax1.twinx()
ax2.plot(time_min, connection_df['control_output'],
         color='red', alpha=0.7, linewidth=0.8)

ax2.set_ylabel('Напряжение на фазовращателе', color='red')

plt.xlim(left=0, right=95)
plt.grid(False)
plt.tight_layout()
plt.savefig('combined_plot.png', dpi=300)
plt.show()

In [ ]:
def hsl_to_rgb_normalized(h, s, l):
    # colorsys использует HLS (не HSL)
    # поэтому порядок: (h, l, s)
    from colorsys import hls_to_rgb
    return hls_to_rgb(h / 360, l / 100, s / 100)

BLUE_RGB = hsl_to_rgb_normalized(206, 73, 48)
GREEN_RGB = hsl_to_rgb_normalized(120, 72, 44)
RED_RGB = hsl_to_rgb_normalized(9, 98, 63)
PURPLE_RGB = hsl_to_rgb_normalized(279, 98, 76)
LIGHT_PURPLE_RGB = hsl_to_rgb_normalized(278, 100, 94)

GRAY_RGB = (123 / 255, 123 / 255, 123 / 255)

plt.rcParams.update({
    "font.size": 24,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 24,
})

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 7))

left = 39

ax1.plot(time_min - left, connection_df['process_variable'] / 10,
            'g-', alpha=0.7, linewidth=1.0, color=BLUE_RGB)
ax1.axhline(y=setpoint, linestyle='--', linewidth=5, color=RED_RGB,
            label=f'Setpoint')
ax1.set_xlabel('Время, мин')
ax1.set_ylabel('Напряжение на фотодетекторе', color=BLUE_RGB)
ax1.set_ylim(top=200)

ax2 = ax1.twinx()
ax2.plot(time_min - left, connection_df['control_output'],
            'r-', alpha=0.7, linewidth=1.0, color=GREEN_RGB)
ax2.set_ylabel('Напряжение на фазовращателе', color=GREEN_RGB)

plt.xlim(left=0, right=42)
plt.grid(False)
plt.tight_layout()
plt.savefig('signals_with_stab.pdf')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(time_min, connection_df['control_output'],
            '-', alpha=0.7, linewidth=1.0, color=GREEN_RGB)
ax.set_ylabel('Напряжение на фазовращателе')

ax.axvline(
    x=switch_time_min,
    color="orange",
    linestyle="--",
    linewidth=5,
    label='Переключение на НС'
)
ax.set_xlabel('Время, мин')
    
plt.xlim(left=0.1)
plt.grid(False)
plt.tight_layout()
plt.savefig('pid_tune_3coef_control_output.pdf')

In [ ]:
neural_network_step = neural_network_step * config.env.args.block_size + len(reset_steps) * 1000

if len(connection_df) > 0:
    setpoint = config.env.args.setpoint

    fig, ax1 = plt.subplots(figsize=(12, 5))

    # Первая ось (Process Variable)
    ax1.plot(connection_df['step'], connection_df['process_variable'],
             'g-', alpha=0.7, linewidth=0.8, label='Process Variable')
    ax1.axhline(y=setpoint, color='black', linestyle='--', alpha=0.6, linewidth=3,
                label=f'Setpoint ({setpoint})')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Process Variable', color='g')
    ax1.set_ylim(500, 1700)

    # Вторая ось (Control Output)
    ax2 = ax1.twinx()
    ax2.plot(connection_df['step'], connection_df['control_output'],
             'r-', alpha=0.7, linewidth=0.8, label='Control Output')
    ax2.set_ylabel('Control Output', color='r')

    plt.xlim(left=10)
    plt.grid(False)
    plt.tight_layout()
    plt.savefig('combined_plot.png')
    plt.show()

In [ ]:
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    "font.size": 24,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 24,
})

columns_left = ['Kp', 'Ki', 'Kd']
colors_left = [GREEN_RGB, BLUE_RGB, RED_RGB]
column_right = 'Error std norm'
column_right_ru = 'Стандартное отклонение\nнормированной ошибки'

time_min = (step_df['time'] - step_df['time'].min()) / 60.0

neural_network_step = max(initial_collect_steps, exploration_steps) - 100 # этап warmup

switch_time_min = None
if neural_network_step <= step_df['Step'].max():
    mask_exact = step_df['Step'] == neural_network_step
    if mask_exact.any():
        switch_time_min = float(time_min[mask_exact].iloc[0])
    else:
        switch_time_min = float(
            np.interp(neural_network_step, step_df['Step'].to_numpy(), time_min.to_numpy())
        )

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(
    3, 2,
    width_ratios=[1, 1.2],
    height_ratios=[1, 1, 1],
    wspace=0.3, hspace=0.4
)

# Левый столбец: 3 графика с общей осью X
axes_left = []
for i, col in enumerate(columns_left):
    if i == 0:
        ax = fig.add_subplot(gs[i, 0])
    else:
        ax = fig.add_subplot(gs[i, 0], sharex=axes_left[0])
    axes_left.append(ax)

    ax.plot(time_min, step_df[col], alpha=0.8, linewidth=1.2, color=colors_left[i])

    if switch_time_min is not None:
        ax.axvline(x=switch_time_min, color='red', linestyle='--', linewidth=2)

    ax.set_ylabel(col)
    ax.grid(False)

# Подпись X только на нижнем графике
axes_left[-1].set_xlabel('Время, мин')
for ax in axes_left[:-1]:
    ax.tick_params(labelbottom=False)

# Правый большой график
ax_right = fig.add_subplot(gs[:, 1])
ax_right.plot(
    time_min,
    step_df[column_right],
    alpha=0.8,
    linewidth=1.5,
    color=PURPLE_RGB,
)

if switch_time_min is not None:
    ax_right.axvline(
        x=switch_time_min,
        color='red',
        linestyle='--',
        linewidth=2,
        label='Переключение на НС'
    )

ax_right.set_xlabel('Время, мин')
ax_right.set_ylabel(column_right_ru)
ax_right.grid(False)

fig.align_ylabels(axes_left)
fig.subplots_adjust(left=0.08, right=0.95, top=0.93, bottom=0.10, wspace=0.3, hspace=0.4)
plt.savefig("pid_adjusting_combine.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    "font.size": 24,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 24,
})

error_mean_normalization_factor = config.env.args.error_mean_normalization_factor
error_std_normalization_factor = config.env.args.error_std_normalization_factor

error_mean = step_df['Error mean norm'] * error_mean_normalization_factor
error_std = step_df['Error std norm'] * error_std_normalization_factor
rmse = np.sqrt(error_mean ** 2 + error_std ** 2) / 10 # старый масштаб

columns_left = ['Kp', 'Ki', 'Kd']
colors_left = [GREEN_RGB, BLUE_RGB, RED_RGB]
column_right_ru = 'Среднеквадратическая\nошибка'

time_min = (step_df['time'] - step_df['time'].min()) / 60.0

switch_time_min = None
if neural_network_step <= step_df['Step'].max():
    mask_exact = step_df['Step'] == neural_network_step
    if mask_exact.any():
        switch_time_min = float(time_min[mask_exact].iloc[0])
    else:
        switch_time_min = float(
            np.interp(neural_network_step, step_df['Step'].to_numpy(), time_min.to_numpy())
        )

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(
    3, 2,
    width_ratios=[1, 1.2],
    height_ratios=[1, 1, 1],
    wspace=0.3, hspace=0.4
)

# Левый столбец: 3 графика с общей осью X
axes_left = []
for i, col in enumerate(columns_left):
    if i == 0:
        ax = fig.add_subplot(gs[i, 0])
    else:
        ax = fig.add_subplot(gs[i, 0], sharex=axes_left[0])
    axes_left.append(ax)

    ax.plot(time_min, step_df[col], alpha=0.8, linewidth=1.2, color=colors_left[i])

    if switch_time_min is not None:
        ax.axvline(x=switch_time_min, color='orange', linestyle='--', linewidth=5)

    ax.set_ylabel(col)
    ax.grid(False)

# Подпись X только на нижнем графике
axes_left[-1].set_xlabel('Время, мин')
for ax in axes_left[:-1]:
    ax.tick_params(labelbottom=False)

# Правый большой график
ax_right = fig.add_subplot(gs[:, 1])
ax_right.plot(
    time_min,
    rmse,
    alpha=0.8,
    linewidth=1.5,
    color=PURPLE_RGB,
)

if switch_time_min is not None:
    ax_right.axvline(
        x=switch_time_min,
        color='orange',
        linestyle='--',
        linewidth=5,
        label='Включение НС-контроллера'
    )

ax_right.set_xlabel('Время, мин')
ax_right.set_ylabel(column_right_ru)
ax_right.grid(False)

handles, labels = ax_right.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.95),
    ncol=2,
    frameon=False,
)

fig.align_ylabels(axes_left)
fig.subplots_adjust(left=0.08, right=0.95, top=0.93, bottom=0.10, wspace=0.3, hspace=0.4)

plt.savefig("pid_tune_3coef.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams.update({'font.size': 20})  

columns_left = ['Kp', 'Ki', 'Kd']
colors_left = ['green', 'blue', 'orange']  
column_right = 'Error std norm'

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(3, 2, width_ratios=[1, 1.2], height_ratios=[1,1,1], wspace=0.3, hspace=0.4)

for i, col in enumerate(columns_left):
    ax = fig.add_subplot(gs[i,0])
    ax.plot(step_df['Step'], step_df[col], alpha=0.8, linewidth=1.2, color=colors_left[i])
    
    if neural_network_step <= step_df['Step'].max():
        ax.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2)
    
    ax.set_ylabel(col)
    ax.set_xlabel('Block')
    ax.grid(True, alpha=0.3)

ax_right = fig.add_subplot(gs[:,1])  
ax_right.plot(step_df['Step'], step_df[column_right], alpha=0.8, linewidth=1.5, color='purple', label=column_right)

if neural_network_step <= step_df['Step'].max():
    ax_right.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, label='Switch to NN')

ax_right.set_xlabel('Block')
ax_right.set_ylabel(column_right)
ax_right.grid(True, alpha=0.3)
ax_right.legend()

fig.subplots_adjust(left=0.08, right=0.95, top=0.93, bottom=0.08, wspace=0.3, hspace=0.4)
plt.show()


In [ ]:
neural_network_step = neural_network_step * config.env.args.block_size + len(reset_steps) * 1000

plt.rcParams.update({'font.size': 14})

columns_left = ['process_variable']
columns_right = ['control_output']
fig = plt.figure(figsize=(18, 6))
gs = GridSpec(1, 2, width_ratios=[1, 1], wspace=0.3)

ax_left = fig.add_subplot(gs[0,0])
ax_left.plot(connection_df['step'], connection_df['process_variable'], 'b-', alpha=0.7, linewidth=1.2, label='Process Variable')
ax_left.axhline(y=setpoint, color='r', linestyle='--', label=f'Setpoint')
ax_left.set_xlim(left=100)

if neural_network_step <= connection_df['step'].max():
    ax_left.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, label='Switch to NN')

ax_left.set_xlabel('Step')
ax_left.set_ylabel('Process Variable')
ax_left.set_title('Process Variable')
ax_left.grid(True, alpha=0.3)
ax_left.legend()

ax_right = fig.add_subplot(gs[0,1])
ax_right.plot(connection_df['step'], connection_df['control_output'], 'g-', alpha=0.7, linewidth=1.2, label='Control Output')
ax_right.set_xlim(left=100)

if neural_network_step <= connection_df['step'].max():
    ax_right.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2)

ax_right.set_xlabel('Step')
ax_right.set_ylabel('Control Output')
ax_right.set_title('Control Output')
ax_right.grid(True, alpha=0.3)
ax_right.legend()

plt.tight_layout()
plt.show()


In [ ]:
plt.rcParams.update({'font.size': 14})

plt.figure(figsize=(12, 5))

plt.plot(connection_df['step'], connection_df['control_output'], 'g-', alpha=0.7, linewidth=1.2, label='Control Output')

if neural_network_step <= connection_df['step'].max():
    plt.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, label='Switch to NN')

plt.xlim(left=100)

plt.xlabel('Step')
plt.ylabel('Control Output')
plt.title('Control Output')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### Эксперимент 2026-01-17_15-18-43 (Kp, Ki, Kd, ICLO)

In [ ]:
plt.rcParams.update({
    'font.size': 16,          # шрифт для подписей осей и текста
    'axes.labelsize': 18,     # шрифт для подписей осей
    'axes.titlesize': 18,     # шрифт заголовка графика
    'xtick.labelsize': 16,    # шрифт делений по X
    'ytick.labelsize': 16,    # шрифт делений по Y
    'legend.fontsize': 16     # шрифт легенды
})

step_df['Connection step'] = step_df['Step'] * block_size
step_df['Error std absolute'] = step_df['Error std norm'] * 200

fig = plt.figure(figsize=(24, 10))
gs = GridSpec(6, 2, figure=fig, hspace=0.8, wspace=0.3, height_ratios=[1, 1, 1, 1, 1, 1])

kp_ax = fig.add_subplot(gs[0:2, 0])   
ki_ax = fig.add_subplot(gs[2:4, 0])   
kd_ax = fig.add_subplot(gs[4:6, 0])

control_ax = fig.add_subplot(gs[0:3, 1])      
error_std_ax = fig.add_subplot(gs[3:6, 1])

kp_ax.plot(step_df['Step'], step_df['Kp'], alpha=0.8, linewidth=0.8, color='green')
kp_ax.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2)
kp_ax.set_ylabel('Kp')
kp_ax.grid(True, alpha=0.3)
kp_ax.set_xlim(right=6_050)
kp_ax.set_xticklabels([]) 

ki_ax.plot(step_df['Step'], step_df['Ki'], alpha=0.8, linewidth=0.8, color='blue')
ki_ax.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2)
ki_ax.set_ylabel('Ki')
ki_ax.grid(True, alpha=0.3)
ki_ax.set_xlim(right=6_050)
ki_ax.set_xticklabels([]) 

kd_ax.plot(step_df['Step'], step_df['Kd'], alpha=0.8, linewidth=0.8, color='orange')
kd_ax.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2)
kd_ax.set_ylabel('Kd')
kd_ax.set_xlabel('NN Step')
kd_ax.grid(True, alpha=0.3)
kd_ax.set_xlim(right=6_050)
kd_ax.set_yticks([0.0025, 0.0020, 0.0015, 0.0010])

control_ax.plot(connection_df['Connection step'], connection_df['Control output'], 
                alpha=0.7, linewidth=0.8, color='brown')
neural_network_connection_step = neural_network_step * block_size
control_ax.axvline(x=neural_network_connection_step, color='red', linestyle='--', linewidth=2, 
                    label=f'Switch NN')
control_ax.set_ylabel('$U_{out}$')
control_ax.set_xlabel('PID Step')
control_ax.grid(True, alpha=0.3)
control_ax.set_xlim(left=10_000)
control_ax.legend(loc='best')

error_std_ax.plot(step_df['Step'], step_df['Error std absolute'], 
                  alpha=0.7, linewidth=0.8, color='purple')
error_std_ax.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2)
error_std_ax.set_ylabel('Error Std')
error_std_ax.set_xlabel('NN Step')
error_std_ax.grid(True, alpha=0.3)
error_std_ax.set_xlim(right=6_050)
error_std_ax.set_ylim(top=40)

fig.tight_layout()

fig.savefig('iclo.eps', format='eps')
fig.savefig('iclo.svg')
fig.savefig('iclo.pdf')